# Identity Matrices Example

This notebook demonstrates isometry pursuit on simulated data consisting of:
- One 10-dimensional identity matrix
- One 8-dimensional identity matrix (embedded in 10D)

Both matrices are embedded in 10-dimensional space, and we run isometry pursuit to see which columns are selected.


In [23]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
from isometry_pursuit.algorithm import isometry_pursuit
from isometry_pursuit.exec.generate_orthogonal_data import generate_orthogonal_data

np.random.seed(1234)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
# Generate data: one 10D identity and one 8D identity embedded in 10D
# matrix_specs: [(10, 1), (8, 1)] means 1 matrix of dimension 10 and 1 matrix of dimension 8
# embedding_dim=10 embeds both in 10-dimensional space
data_tensor, metadata = generate_orthogonal_data(
    output_dir="data/identity_example",
    matrix_specs=[(10, 1), (8, 1)],
    n_replicates=1,
    embedding_dim=10,
    noise_level=0.0,
    seed=1234
)

print(f"Data tensor shape: {data_tensor.shape}")
print(f"Metadata: {metadata}")

# Extract the first (and only) replicate as our data matrix X
# Shape: (D, n_samples) = (10, 18) - 10 columns from 10D identity + 8 columns from 8D identity
X = data_tensor[0, :, :]
print(f"\nData matrix X shape: {X.shape}")
print(f"Expected: (10, 18) - 10 columns from 10D identity + 8 columns from 8D identity")


Generating orthogonal matrices...
Generating 1 10-dimensional orthogonal matrices...
  Matrix 1: shape (10, 10)
Generating 1 8-dimensional orthogonal matrices...
  Matrix 1: shape (10, 8)
Total matrices: 2
Total samples: 18 (dimension samples per matrix)
Generating 1 replicates...
Generated data tensor shape: (1, 10, 18)
Data tensor saved to data/identity_example/orthogonal_data_tensor.npy
Metadata saved to data/identity_example/orthogonal_data_metadata.json
Data tensor shape: (1, 10, 18)
Metadata: {'dataset_name': 'orthogonal_simulated', 'matrix_specs': [(10, 1), (8, 1)], 'n_matrices_total': 2, 'n_samples_total': 18, 'n_replicates': 1, 'dimension': 10, 'noise_level': 0.0, 'seed': 1234, 'data_file': 'data/identity_example/orthogonal_data_tensor.npy', 'description': 'Simulated data with orthogonal matrices: [(10, 1), (8, 1)], all embedded in 10D space, dimension samples per matrix'}

Data matrix X shape: (10, 18)
Expected: (10, 18) - 10 columns from 10D identity + 8 columns from 8D iden

In [25]:
# Inspect the data matrix
print("First 10 columns (from 10D identity):")
print(X[:, :10])
print("\nNext 8 columns (from 8D identity embedded in 10D):")
print(X[:, 10:])
print("\nColumn norms:")
for i in range(X.shape[1]):
    norm = np.linalg.norm(X[:, i])
    print(f"  Column {i}: {norm:.6f}")


First 10 columns (from 10D identity):
[[ 0.46956888  0.12755694 -0.3882532   0.05469149 -0.13972845  0.10074308
   0.4687598  -0.41090038  0.37258102 -0.22899322]
 [-0.2412042  -0.48058605 -0.17036582  0.215551    0.22401142 -0.22183492
  -0.12958643 -0.30555996 -0.15950115 -0.63275496]
 [-0.09555773  0.21861757 -0.26199748  0.47438009  0.0147127   0.51547993
  -0.08150499  0.52121986  0.04947835 -0.3204662 ]
 [-0.48291858  0.20122756  0.4953539  -0.05447723  0.09833425  0.40974422
   0.10201901 -0.38980075  0.34075963 -0.14806849]
 [-0.01633941 -0.2922513  -0.20692462 -0.35812612 -0.01051629  0.63096957
   0.17733793 -0.14076771 -0.53215442  0.10278267]
 [ 0.4261906   0.41486396  0.12340684 -0.3288875   0.1281482   0.09234769
  -0.54483793 -0.13550321 -0.1811114  -0.38715932]
 [ 0.00361187  0.15828972  0.08276406  0.5455046  -0.53138232  0.03450814
  -0.21007999 -0.42736828 -0.35689926  0.18109071]
 [-0.48578249  0.47510066 -0.32192819 -0.32831743 -0.31984297 -0.28394109
   0.2044654 

In [30]:
X.shape

(10, 18)

In [29]:
# Run isometry pursuit on the data
result = isometry_pursuit(X, power=1.0)

beta = result["beta"]
selected_indices = result["basis_pursuit_indices"]

print("Isometry Pursuit Results:")
print(f"Selected indices: {selected_indices}")
print(f"Number of selected columns: {len(selected_indices)}")

for i in range(beta.shape[0]):
    norm = np.linalg.norm(beta[i, :])
    if norm > 1e-6:  # Only print non-zero rows
        print(f"  Row {i}: {norm:.6f}")



Isometry Pursuit Results:
Selected indices: [0 1 2 3 4 5 6 7 8 9]
Number of selected columns: 10
  Row 0: 1.000000
  Row 1: 1.000000
  Row 2: 1.000000
  Row 3: 1.000000
  Row 4: 1.000000
  Row 5: 1.000000
  Row 6: 1.000000
  Row 7: 1.000000
  Row 8: 1.000000
  Row 9: 1.000000


/Users/samsonkoelle/miniconda3/envs/isometry_pursuit/lib/python3.11/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


In [ ]:
# Analyze which columns were selected
print("Analysis of selected columns:")
print(f"\nSelected indices: {selected_indices}")

# Check which come from the 10D identity (indices 0-9)
from_10d = [idx for idx in selected_indices if idx < 10]
# Check which come from the 8D identity (indices 10-17)
from_8d = [idx for idx in selected_indices if idx >= 10]

print(f"\nColumns from 10D identity (indices 0-9): {from_10d} ({len(from_10d)} columns)")
print(f"Columns from 8D identity (indices 10-17): {from_8d} ({len(from_8d)} columns)")

# Check if we got exactly 10 columns (the dimension)
print(f"\nExpected: 10 columns (since D=10)")
print(f"Actual: {len(selected_indices)} columns")


In [ ]:
# Verify the constraint: w(X) @ beta should equal I_D
from isometry_pursuit.transformation import exponential_transformation

X_transformed = exponential_transformation(X, power=1.0)
constraint_check = X_transformed @ beta

print("Constraint verification (w(X) @ beta should equal I_10):")
print(f"Error (should be close to zero): {np.linalg.norm(constraint_check - np.eye(10), 'fro'):.2e}")
print(f"\nFirst few rows/columns of constraint check:")
print(constraint_check[:5, :5])


In [ ]:
# Visualize the beta coefficients
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Beta row norms (coefficient magnitudes)
ax1 = axes[0]
row_norms = [np.linalg.norm(beta[i, :]) for i in range(beta.shape[0])]
colors = ['red' if i < 10 else 'blue' for i in range(len(row_norms))]
bars = ax1.bar(range(len(row_norms)), row_norms, color=colors, alpha=0.7)
ax1.set_xlabel('Column Index')
ax1.set_ylabel('||β_{p.}||_2')
ax1.set_title('Beta Row Norms (Solution Coefficients)')
ax1.set_xticks(range(len(row_norms)))
ax1.set_xticklabels([f'{i}' for i in range(len(row_norms))], rotation=45)
ax1.axvline(x=9.5, color='gray', linestyle='--', linewidth=1, label='10D/8D boundary')
ax1.legend(['10D identity', '8D identity', 'Boundary'])
ax1.grid(True, alpha=0.3, axis='y')

# Plot 2: Selected vs non-selected columns
ax2 = axes[1]
selected_binary = [1 if i in selected_indices else 0 for i in range(beta.shape[0])]
colors2 = ['green' if i in selected_indices else 'gray' for i in range(len(selected_binary))]
bars2 = ax2.bar(range(len(selected_binary)), selected_binary, color=colors2, alpha=0.7)
ax2.set_xlabel('Column Index')
ax2.set_ylabel('Selected (1) or Not (0)')
ax2.set_title('Selected Columns')
ax2.set_xticks(range(len(selected_binary)))
ax2.set_xticklabels([f'{i}' for i in range(len(selected_binary))], rotation=45)
ax2.axvline(x=9.5, color='gray', linestyle='--', linewidth=1)
ax2.set_ylim(-0.1, 1.1)
ax2.legend(['Selected', 'Not selected', 'Boundary'])
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


In [ ]:
# Check if the selected columns form an orthonormal basis
if len(selected_indices) == 10:
    selected_cols = X[:, selected_indices]
    gram_matrix = selected_cols.T @ selected_cols
    print("Gram matrix of selected columns (should be identity if orthonormal):")
    print(gram_matrix)
    print(f"\nIs identity? {np.allclose(gram_matrix, np.eye(10))}")
    print(f"Max deviation from identity: {np.max(np.abs(gram_matrix - np.eye(10))):.2e}")
else:
    print(f"Selected {len(selected_indices)} columns, but need exactly 10 for an orthonormal basis in 10D")
